In [1]:
# Modules
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, rc
from IPython.display import HTML
# Mounting Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Plotting figures. Overlaying multiple graphs in a single figure
def plot_solution(xgrid, uvect):
  # Setting parameters related to plotting
  umax = np.max(sol)
  umin = np.min(sol)
  ulength = umax - umin
  umax += 0.05 * ulength
  umin -= 0.05 * ulength
  # t = 0
  plt.plot(xgrid,uvect[0,:],color='#ff8c00')
  # t > 0
  for n in range(1,uvect.shape[0]):
    plt.plot(xgrid,uvect[n,:],color='#00bfff')

  plt.xlabel('x')
  plt.ylabel('u')
  plt.ylim(umin, umax)
  plt.grid('on')
  plt.show()

In [3]:
# Creating animations
# Using the IPython.display module
def plot_animation(xgrid, tgrid, uvect):
  # Configuring plotting-related parameters
  umax = np.max(sol)
  umin = np.min(sol)
  xmax = np.max(xgrid)
  xmin = np.min(xgrid)
  ulength = umax - umin
  utop = umax + 0.1*ulength
  umax += 0.05*ulength
  umin -= 0.05*ulength
  xmid = (xmax+xmin)/2
  xlength = xmax - xmin
  xmax += 0.05*xlength
  xmin -= 0.05*xlength

  # Creating fig and ax objects
  fig, ax = plt.subplots()
  ims = []

  # t = 0
  n = 0
  im, = ax.plot(xgrid, uvect[n, :], color='#ff8c00')
  title = ax.text(xmid, utop, f'time={tgrid[n]:.4f}', ha='center', va='center', fontsize=15)
  ims.append([im, title])

  # t > 0
  for n in range(1, uvect.shape[0]):
    im, = ax.plot(xgrid, uvect[n, :], color='#00bfff')
    title = ax.text(xmid, utop, f'time={tgrid[n]:.4f}', ha='center', va='center', fontsize=15)
    ims.append([im, title])

  # Setting labels for each axis
  ax.set_xlabel(r"$x$", fontsize=15)
  ax.set_ylabel(r"$u$", fontsize=15)
  # Setting the range of the graph
  ax.set_xlim([xmin, xmax])
  ax.set_ylim([umin, umax])
  ax.grid(True)

  # Creating an animation by passing the fig object and ims to ArtistAnimation
  return animation.ArtistAnimation(fig, ims)

In [4]:
# Central difference scheme for the linear concevtion equation
def linear_convection_central(c, initial, N, T, mu, num):
  L = 1.0
  h = L/N
  tau = mu*h/np.abs(c)
  lam = tau/h
  nmax = int(T/tau)
  step = int(max(1, nmax/num))

  xgrid = np.linspace(0.0, L, N)
  xtmp = np.copy(xgrid[0:N-1])
  tgrid = np.array([0.0])
  u = initial(xtmp)
  sol = np.copy(np.append(u,u[0]))

  for n in range(nmax):
    u = u - 0.5*c*lam*(np.roll(u,-1)-np.roll(u,1))
    tnow = (n+1)*tau
    if (n+1)%step==0:
      sol=np.vstack((sol,np.append(u,u[0])))
      tgrid=np.append(tgrid, tnow)

  return xgrid, tgrid, sol

In [5]:
# Upwind difference scheme for the linear concevtion equation
def linear_convection_upwind(c, initial, N, T, mu, num):
  L = 1.0
  h = L/N
  tau = mu*h/np.abs(c)
  nmax = int(T/tau)
  step = int(max(1, nmax/num))
  lam = tau/h
  theta = 1 if c >= 0 else 0

  xgrid = np.linspace(0.0, L, N)
  xtmp = np.copy(xgrid[0:N-1])
  tgrid = np.array([0.0])
  u = initial(xtmp)
  sol = np.copy(np.append(u,u[0]))

  for n in range(nmax):
    p = 1.0 - theta*c*lam + (1.0 - theta)*c*lam
    q = theta*c*lam
    r = (1.0 - theta)*c*lam
    u = p*u + q*np.roll(u, 1) - r*np.roll(u, -1)
    tnow = (n+1)*tau
    if (n+1)%step==0:
      sol=np.vstack((sol,np.append(u,u[0])))
      tgrid=np.append(tgrid, tnow)

  return xgrid, tgrid, sol

In [6]:
# Lax--Friedrics difference scheme for the linear concevtion equation
def linear_convection_lf(c, initial, N, T, mu, num):
  L = 1.0
  h = L/N
  tau = mu*h/np.abs(c)
  nmax = int(T/tau)
  step = int(max(1, nmax/num))
  lam = tau/h

  xgrid = np.linspace(0.0, L, N)
  xtmp = np.copy(xgrid[0:N-1])
  tgrid = np.array([0.0])
  u = np.vectorize(initial)(xtmp)
  sol = np.copy(np.append(u,u[0]))

  for n in range(nmax):
    u = 0.5*(1.0 + c*lam)*np.roll(u,1) + 0.5*(1.0 - c*lam)*np.roll(u,-1)
    tnow = (n+1)*tau
    if (n+1)%step==0:
      sol=np.vstack((sol,np.append(u,u[0])))
      tgrid=np.append(tgrid, tnow)

  return xgrid, tgrid, sol

In [ ]:
def initial1(x):
  return np.sin(4*np.pi*x)

def initial2(x):
  return  np.maximum(0.0, np.minimum(2.0*x-0.5, 1.5-2.0*x))

# Setting parameters
c = -1.0
N = 51
T = 1.0
mu = 1
num = 40

# Computing the linear convection equation
x, tn, sol = linear_convection_central(c, initial1, N, T, mu, num)
#x, tn, sol = linear_convection_upwind(c, initial1, N, T, mu, num)
#x, tn, sol = linear_convection_lf(c, initial1, N, T, mu, num)

# Displaying figures
plot_solution(x, sol)

# Displaying an animation
anim = plot_animation(x, tn, sol)
# Saving the results as convect.gif (make sure that Google Drive is mounted)
anim.save('/content/drive/MyDrive/Colab Notebooks/fig/convect.gif', writer='pillow')
rc('animation', html='jshtml')
plt.close()
anim